# 🌿 Entraînement EfficientNet-B3 — Détection de Plantes PLBD-36
**Groupe 36 | École Centrale Casablanca**

Ce notebook entraîne un modèle de classification de plantes via **Transfer Learning** avec **EfficientNet-B3** sur le dataset **PlantVillage**.

## Pipeline complet
1. Installation des dépendances
2. Téléchargement du dataset PlantVillage (KaggleHub)
3. Préparation des données (split 70/15/15)
4. Entraînement EfficientNet-B3 (Transfer Learning)
5. Évaluation sur le test set + matrice de confusion
6. Export PyTorch (`.pth`) pour développement
7. Export TFLite (`.tflite`) pour déploiement Raspberry Pi
8. Génération du mapping besoins hydriques par espèce
9. Test de démonstration d'inférence

## 1. Installation des dépendances

In [ ]:
# Installation des dépendances
# À exécuter une seule fois (décommenter si nécessaire)
# !pip install kagglehub torch torchvision efficientnet_pytorch
# !pip install tensorflow onnx onnx-tf scikit-learn matplotlib seaborn

import os, json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
print('Imports de base OK')

## 2. Téléchargement du dataset PlantVillage

In [ ]:
import kagglehub

# Téléchargement automatique du dataset PlantVillage
# (~2 GB, contient ~87 000 images de 38 classes de plantes/maladies)
path = kagglehub.dataset_download('abdallahalidev/plantvillage-dataset')
print(f'Dataset téléchargé : {path}')

# Trouver le dossier contenant les images
for root, dirs, files in os.walk(path):
    if any(f.endswith(('.jpg','.JPG','.png')) for f in files):
        DATA_DIR = root
        break

print(f'Dossier images : {DATA_DIR}')
classes = sorted(os.listdir(DATA_DIR))
print(f'Nombre de classes : {len(classes)}')
print('Classes disponibles :')
for i, c in enumerate(classes): print(f'  {i:2d}. {c}')

## 3. Préparation des données (DataLoaders PyTorch)

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# --- Configuration ---
BATCH_SIZE   = 32
IMAGE_SIZE   = 300   # EfficientNet-B3 input
NUM_WORKERS  = 2
SEED         = 42
torch.manual_seed(SEED)

# --- Transformations ---
transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# --- Chargement dataset ---
full_dataset = datasets.ImageFolder(DATA_DIR, transform=transform_train)
NUM_CLASSES  = len(full_dataset.classes)
print(f'Total images : {len(full_dataset)} | Classes : {NUM_CLASSES}')

# --- Split 70/15/15 ---
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

# Appliquer transform_val sur val et test
val_ds.dataset.transform  = transform_val
test_ds.dataset.transform = transform_val

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
CLASS_NAMES = full_dataset.classes
print(f'Exemple de classes : {CLASS_NAMES[:5]}...')

## 4. Modèle EfficientNet-B3 (Transfer Learning)

In [ ]:
import torch.nn as nn
from torchvision import models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

# --- Modèle EfficientNet-B3 pré-entraîné ImageNet ---
model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)

# Geler les couches de base (transfer learning)
for param in model.parameters():
    param.requires_grad = False

# Remplacer le classifier pour notre nombre de classes
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)
print(f'Modèle EfficientNet-B3 prêt ({NUM_CLASSES} classes)')
print(f'Paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5. Entraînement

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

# --- Hyperparamètres ---
EPOCHS    = 15
LR        = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.classifier.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

# --- Phase de fine-tuning : dégeler les derniers blocs après 5 epochs ---
UNFREEZE_EPOCH = 5

historique_entrainement = []

def evaluer(loader):
    model.eval()
    correct, total, loss_total = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss_total += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return loss_total / len(loader), correct / total

print('Debut de l entraînement...')
meilleure_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # Dégeler les dernières couches à UNFREEZE_EPOCH
    if epoch == UNFREEZE_EPOCH:
        for name, param in model.named_parameters():
            if 'features.8' in name or 'features.7' in name or 'classifier' in name:
                param.requires_grad = True
        optimizer.add_param_group({'params': [p for p in model.parameters() if p.requires_grad], 'lr': LR * 0.1})
        print(f'[Epoch {epoch}] Fine-tuning active sur derniers blocs')

    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        _, predicted   = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()
        train_total   += labels.size(0)
        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}')

    scheduler.step()
    train_acc = train_correct / train_total
    val_loss, val_acc = evaluer(val_loader)

    historique_entrainement.append({'epoch': epoch, 'train_loss': train_loss/len(train_loader),
                                     'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})

    print(f'Epoch {epoch:2d}/{EPOCHS} | Train acc: {train_acc:.3f} | Val acc: {val_acc:.3f} | Val loss: {val_loss:.4f}')

    if val_acc > meilleure_val_acc:
        meilleure_val_acc = val_acc
        torch.save(model.state_dict(), 'modele/best_weights.pth')
        print(f'  -> Meilleur modèle sauvegardé (val_acc={val_acc:.3f})')

print(f'\nEntraînement terminé | Meilleure val acc : {meilleure_val_acc:.3f}')

## 6. Évaluation sur le test set + Matrice de confusion

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Charger les meilleurs poids
model.load_state_dict(torch.load('modele/best_weights.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc   = (all_preds == all_labels).mean()

print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print('\nRapport par classe :')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

# --- Matrice de confusion ---
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_xlabel('Prédit', fontsize=12)
ax.set_ylabel('Réel', fontsize=12)
ax.set_title(f'Matrice de Confusion — Test Accuracy : {test_acc:.2%}', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig('modele/matrice_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Matrice sauvegardée : modele/matrice_confusion.png')

## 7. Export PyTorch complet (.pth)

In [ ]:
# Sauvegarde checkpoint complet (architecture + poids + métadonnées)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'image_size': IMAGE_SIZE,
    'architecture': 'efficientnet_b3',
    'test_accuracy': float(test_acc),
    'epochs_trained': EPOCHS
}
torch.save(checkpoint, 'modele/efficientnet_b3_plants_complet.pth')
print('Checkpoint PyTorch sauvegardé : modele/efficientnet_b3_plants_complet.pth')

# Sauvegarde classes.txt
with open('modele/classes.txt', 'w') as f:
    for c in CLASS_NAMES:
        f.write(c + '\n')
print(f'Classes sauvegardées : modele/classes.txt ({len(CLASS_NAMES)} classes)')

## 8. Export TFLite pour Raspberry Pi (PyTorch → ONNX → TF → TFLite)

In [ ]:
import torch

# Étape 1 : Export ONNX
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
model.eval()

torch.onnx.export(
    model, dummy_input,
    'modele/efficientnet_b3.onnx',
    export_params=True,
    opset_version=12,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)
print('Export ONNX OK : modele/efficientnet_b3.onnx')

# Étape 2 : ONNX → TensorFlow
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load('modele/efficientnet_b3.onnx')
tf_rep = prepare(onnx_model)
tf_rep.export_graph('modele/efficientnet_b3_tf')
print('Export TF OK : modele/efficientnet_b3_tf/')

# Étape 3 : TF → TFLite avec quantification float16
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model('modele/efficientnet_b3_tf')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('modele/efficientnet_b3_plants.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f'Export TFLite OK : modele/efficientnet_b3_plants.tflite ({size_mb:.1f} MB)')
print('Ce fichier est pret pour le deploiement sur Raspberry Pi Zero 2 W')

## 9. Mapping besoins hydriques par espèce

In [ ]:
# Mapping classes PlantVillage -> besoins hydriques
# Culture de reference : TOMATE (agriculteur enquete au Maroc)

def get_besoins_pour_classe(nom_classe):
    """Retourne les besoins hydriques selon la classe PlantVillage."""
    nom = nom_classe.lower()

    # TOMATE — culture de reference (enquete terrain Maroc)
    if 'tomato' in nom:
        if 'healthy' in nom:
            return {'espece': 'tomate', 'etat': 'saine',
                    'humidite_min': 60, 'humidite_max': 80,
                    'duree_arrosage': 10, 'intervalle_min': 3600}
        else:
            # Tomate malade : moins d'eau pour limiter la propagation fongique
            return {'espece': 'tomate', 'etat': 'malade',
                    'humidite_min': 55, 'humidite_max': 70,
                    'duree_arrosage': 7, 'intervalle_min': 4800}

    # POIVRON
    if 'pepper' in nom:
        return {'espece': 'poivron', 'etat': 'healthy' if 'healthy' in nom else 'malade',
                'humidite_min': 55, 'humidite_max': 75,
                'duree_arrosage': 8, 'intervalle_min': 3600}

    # POMME DE TERRE
    if 'potato' in nom:
        return {'espece': 'pomme_de_terre', 'etat': 'healthy' if 'healthy' in nom else 'malade',
                'humidite_min': 50, 'humidite_max': 75,
                'duree_arrosage': 12, 'intervalle_min': 3600}

    # MAIS
    if 'corn' in nom or 'maize' in nom:
        return {'espece': 'mais', 'etat': 'healthy' if 'healthy' in nom else 'malade',
                'humidite_min': 45, 'humidite_max': 70,
                'duree_arrosage': 15, 'intervalle_min': 7200}

    # RAISIN
    if 'grape' in nom:
        return {'espece': 'vigne', 'etat': 'healthy' if 'healthy' in nom else 'malade',
                'humidite_min': 40, 'humidite_max': 65,
                'duree_arrosage': 10, 'intervalle_min': 7200}

    # Defaut
    return {'espece': 'generique', 'etat': 'unknown',
            'humidite_min': 50, 'humidite_max': 75,
            'duree_arrosage': 10, 'intervalle_min': 3600}

# Construire le mapping complet
besoins_mapping = {}
for cls in CLASS_NAMES:
    besoins_mapping[cls] = get_besoins_pour_classe(cls)

# Sauvegarder
with open('modele/besoins_hydriques.json', 'w', encoding='utf-8') as f:
    json.dump(besoins_mapping, f, ensure_ascii=False, indent=2)

print(f'Mapping besoins hydriques sauvegardé : modele/besoins_hydriques.json')
print(f'Nombre d entrees : {len(besoins_mapping)}')
print('\nExemple (Tomato___healthy) :')
print(json.dumps(besoins_mapping.get('Tomato___healthy', {}), indent=2))

## 10. Test de démonstration d'inférence

In [ ]:
# Test du modele TFLite sur des images du test set
try:
    import tflite_runtime.interpreter as tflite
except ImportError:
    import tensorflow as tf
    tflite = tf.lite

interpreter = tflite.Interpreter(model_path='modele/efficientnet_b3_plants.tflite')
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def preprocess(img_tensor):
    # Dénormaliser depuis ImageNet puis renormaliser pour TFLite
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    img = img_tensor * std + mean
    img = img.permute(1,2,0).numpy()
    img = (img - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
    return img.astype(np.float32)[np.newaxis]

# Selectionner 6 images aleatoires du test set
import random
indices = random.sample(range(len(test_ds)), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Test d inférence TFLite — PLBD-36', fontsize=14)

for i, idx in enumerate(indices):
    image, label_idx = test_ds[idx]
    inp = preprocess(image)
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])[0]
    pred_idx  = int(np.argmax(output))
    confiance = float(output[pred_idx])
    besoins   = besoins_mapping.get(CLASS_NAMES[pred_idx], {})

    # Affichage
    ax = axes[i // 3][i % 3]
    img_show = image.permute(1,2,0).numpy()
    img_show = img_show * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
    img_show = np.clip(img_show, 0, 1)
    ax.imshow(img_show)

    correct = pred_idx == label_idx
    couleur = 'green' if correct else 'red'
    ax.set_title(
        f'Prédit : {CLASS_NAMES[pred_idx]}\n'
        f'Réel : {CLASS_NAMES[label_idx]}\n'
        f'Conf : {confiance:.0%} | Arrosage : {besoins.get("duree_arrosage","?") }s',
        color=couleur, fontsize=8
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('modele/demo_inference.png', dpi=120, bbox_inches='tight')
plt.show()
print('Demo sauvegardée : modele/demo_inference.png')

## ✅ Récapitulatif des fichiers générés

| Fichier | Usage |
|---------|-------|
| `modele/efficientnet_b3_plants_complet.pth` | Développement PyTorch |
| `modele/efficientnet_b3_plants.tflite` | **Déploiement Raspberry Pi** |
| `modele/classes.txt` | Labels pour le robot |
| `modele/besoins_hydriques.json` | Seuils d'arrosage par espèce |
| `modele/matrice_confusion.png` | Visualisation performances |
| `modele/demo_inference.png` | Démonstration inférence |

**Prochaine étape :** Copier `efficientnet_b3_plants.tflite`, `classes.txt` et `besoins_hydriques.json` sur votre Raspberry Pi dans le dossier `modele/` du projet, puis lancer :
```bash
python scripts/inference_test.py
```